# MCC Training (Notebook)

Notebook version of the MCC sentence-level training script.

## Steps
1. Run the setup cell to import dependencies.
2. Set the config parameters.
3. Run the training cell.

In [ ]:
import sys
from pathlib import Path

# Ensure project src/ is on the import path
project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Using project root: {project_root}")
print(f"Added to sys.path: {src_path}")

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Tuple

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

from mcc.data import build_dataloader
from mcc.models import BertMCCClassifier, MCCModelConfig

In [ ]:
# === Configuration ===
TRAIN_PATH = Path("../src/data/mcc_train.jsonl")
DEV_PATH = Path("../src/data/mcc_dev.jsonl")
EPOCHS = 2
BATCH_SIZE = 8
LR = 5e-5
CHECKPOINT_PATH = Path("../src/checkpoints/mcc_bert.pt")
MODEL_NAME = "bert-base-uncased"

In [ ]:
def train_epoch(model, dataloader: DataLoader, optimizer, scheduler, device) -> float:
    model.train()
    total_loss = 0.0
    for batch in tqdm(dataloader, desc="train", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item()
    return total_loss / max(len(dataloader), 1)


def evaluate(model, dataloader: DataLoader, device) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="eval", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits
            preds = logits.argmax(dim=-1)
            total_loss += loss.item()
            correct += (preds == batch["labels"]).sum().item()
            total += preds.numel()
    acc = correct / total if total else 0.0
    return total_loss / max(len(dataloader), 1), acc

In [ ]:
# === Training ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = build_dataloader(TRAIN_PATH, tokenizer, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = build_dataloader(DEV_PATH, tokenizer, batch_size=BATCH_SIZE)

config = MCCModelConfig(pretrained_model_name=MODEL_NAME, num_labels=3)
model = BertMCCClassifier.from_config(config).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * max(EPOCHS, 1)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, total_steps)

best_acc = 0.0
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    dev_loss, dev_acc = evaluate(model, dev_loader, device)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f} dev_loss={dev_loss:.4f} dev_acc={dev_acc:.4f}")
    if dev_acc > best_acc:
        best_acc = dev_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"Saved checkpoint to {CHECKPOINT_PATH}")